<a href="https://colab.research.google.com/github/PeesapatiRohithSharma/Summer_2026/blob/main/random_forests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import numpy as np
import matplotlib.pyplot as plt

In [7]:
class Node:
  def __init__(self,gini, num_samples,num_samples_per_class, predicted_classs):
    self.gini = gini
    self.num_samples = num_samples
    self.per_class = num_samples_per_class
    self.predicted_classs = predicted_classs
    self.threshold = 0
    self.feature_index = 0
    self.left = None
    self.right = None


In [8]:
class Decision_tree:
  def __init__(self,max_depth,min_samples):
    self.max_depth = max_depth
    self.min_samples = min_samples
    self.root = None
    self.num_classes = None

  def gini(self, y):
    return 1.0 - sum((np.sum(y == c)/len(y))**2 for c in set(y))

  def best_split(self, X, y):
    m,n = X.shape
    if m <= self.min_samples:
      return None, None
    else:
      best_gini = self.gini(y)
      best_idx, best_thres = None, None
      for feature_index in range(n):
        for threshold in np.unique(X[:,feature_index]):
          left_indices = X[:,feature_index] <= threshold
          right_indices = ~left_indices
          y_left = y[left_indices]
          y_right = y[right_indices]
          if len(y_left) == 0 or len(y_right) == 0:
            continue
          left_gini = self.gini(y_left)
          right_gini = self.gini(y_right)
          weighted_gini = (len(y_left)/len(y))*(left_gini) + (len(y_right)/len(y))*(right_gini)
          if weighted_gini < best_gini:
            best_gini = weighted_gini
            best_idx = feature_index
            best_thres = threshold
      return best_idx, best_thres

  def grow_tree(self, X, y, depth):
    num_samples_per_class = [np.sum((y==i)) for i in range(self.num_classes)]
    pred_class = np.argmax(num_samples_per_class)
    node = Node(self.gini(y),len(y), num_samples_per_class,pred_class)
    idx,t = self.best_split(X,y)
    if depth < self.max_depth or idx is None:
      left_ind = X[:,idx] <= t
      X_left, y_left = X[left_ind], y[left_ind]
      X_right, y_right = X[~left_ind], y[~left_ind]
      node.feature_index = idx
      node.threshold = t
      node.left = self.grow_tree(X_left, y_left, depth+1)
      node.right = self.grow_tree(X_right, y_right, depth+1)
    return node

  def predict(self,input):
    node = self.tree
    while node.left:
      if input[node.feature_index] < node.threshold:
        node = node.left
      else:
        node = node.right
    return node.pred_class

  def fit(self, X,y):
    self.num_classes = len(np.unique(y))
    self.root = self.grow_tree(X, y , 0)

In [9]:
class Random_Forest:
  def __init__(self, k, max_depth, m, min_samples):
    self.k = k
    self.max_depth = max_depth
    self.m = m
    self.min_samples = min_samples
    self.trees = []

  def boot_strapping(self,X,y):
    boot_strapped = []
    for _ in range(self.m):
      boot_strap_idx = np.random.choice(X.shape[0], X.shape[0], True)
      boot_strapped.append(boot_strap_idx)
      X_boot_strapped = X[boot_strap_idx]
      y_boot_strapped = y[boot_strap_idx]

      feature_index = np.random.choice(X.shape[1], self.k, False)

      X_boot_strap = X_boot_strapped[:,feature_index]
      tree = Decision_tree(max_depth=self.max_depth, min_samples=self.min_samples)
      tree.fit(X_boot_strap, y_boot_strapped)
      self.trees.append((tree, feature_index))


  def predict(self,X):
    predictions = []
    for tree, feature_index in self.trees:
      X_subset = X[:, feature_index]
      preds = [tree.predict(sample) for sample in X_subset]
      predictions.append(preds)
    predictions = np.array(predictions)
    final = []
    for column in predictions.T:
      vals, counts = np.unique(column, return_counts=True)
      final.append(vals[np.argmax(counts)])
    return np.array(final)